# 08 — Routes and buffers (Stage 5)

Two parallel exposure footprints, ready for Stage 6 to count salons in:

1. **Walking routes** — for each `(LSOA, school)` row in the Stage-4 hard-nearest assignment, the shortest walking-network path as a LineString, buffered at **50 m** (primary, spec §6.3) and **100 m** (sensitivity #6).
2. **Euclidean school buffers** — per school, polygon buffers at **400 / 800 / 1600 m** (spec §6.2). The conventional H2 comparator.

Network-distance school service-area buffers are not produced here — for the H2 contrast the Euclidean buffer is the standard comparator and adding network-distance school buffers requires a per-school Dijkstra over the 1.5 M-node graph that adds little H2 signal. They can be added as a sensitivity.

Cross-check (sensitivity #8): a 10% random sample of routes is re-routed via osmnx + networkx Dijkstra and the lengths are compared.

## 0. Pre-flight

In [ ]:
import time
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from schools_sunbeds import audit, config, network as nw, routing

config.ensure_dirs()
audit.verify_manifest()

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
TODAY

## 1. Load network + assignments + schools + PWCs

In [ ]:
net_dir = sorted(p for p in config.DATA_PROCESSED.glob("walk_network_*") if p.is_dir())[-1]
data = nw.load_walking_network(net_dir)
print(f"Network: {len(data.nodes):,} nodes, {len(data.edges):,} edges")

t0 = time.time()
net = nw.build_pandana_network(data)
print(f"pandana.Network ready in {time.time()-t0:.1f}s")

In [ ]:
assignments_path = sorted(config.DATA_PROCESSED.glob("assignments_nearest_*.parquet"))[-1]
assignments = pd.read_parquet(assignments_path)
print(f"Loaded {assignments_path.name}: {len(assignments)} assignments")

schools_path = sorted(
    p for p in config.DATA_PROCESSED.glob("schools_ne_*.gpkg") if "sensitivity" not in p.name
)[-1]
schools = gpd.read_file(schools_path)

pwc_path = sorted(config.DATA_PROCESSED.glob("lsoa_pwc_ne_*.gpkg"))[-1]
pwc = gpd.read_file(pwc_path)
print(f"Schools: {len(schools)} | PWCs: {len(pwc)}")

## 2. Compute walking-route LineStrings

Pandana's `shortest_paths` returns the node sequence per (origin, dest) pair; we materialise each as a BNG LineString. Routes that don't path (origin/dest unreachable, or sequence < 2 nodes) drop out.

In [ ]:
t0 = time.time()
routes = routing.compute_routes(net, assignments, pwc, schools, walk_nodes=data.nodes)
print(f"Routes built in {time.time()-t0:.1f}s; {len(routes)} routes (of {len(assignments)} assignments)")
print("Length stats (m):")
print(routes.groupby("phase_cat")["length_m"].describe().round(0).to_string())

## 3. Route buffers (50 m primary, 100 m sensitivity)

In [ ]:
t0 = time.time()
route_buf_50 = routing.buffer_routes(routes, buffer_m=config.ROUTE_BUFFER_PRIMARY_M)
route_buf_100 = routing.buffer_routes(routes, buffer_m=config.ROUTE_BUFFER_SENSITIVITY_M)
print(f"Buffered in {time.time()-t0:.1f}s")
print(f"  50 m buffer: {len(route_buf_50)} polygons")
print(f"  100 m buffer: {len(route_buf_100)} polygons")

## 4. School Euclidean buffers (400 / 800 / 1600 m)

In [ ]:
t0 = time.time()
school_bufs = routing.school_euclidean_buffers(schools, distances_m=config.BUFFER_DISTANCES_M)
print(f"Built {len(school_bufs)} school-buffer polygons in {time.time()-t0:.1f}s")
school_bufs.groupby("distance_m").size().to_frame("n_buffers")

## 5. Persist outputs

In [ ]:
out_routes = config.DATA_PROCESSED / f"routes_{TODAY}.gpkg"
out_buf50 = config.DATA_PROCESSED / f"route_buffer_50m_{TODAY}.gpkg"
out_buf100 = config.DATA_PROCESSED / f"route_buffer_100m_{TODAY}.gpkg"
out_school_bufs = config.DATA_PROCESSED / f"school_euclidean_buffers_{TODAY}.gpkg"

routes.to_file(out_routes, driver="GPKG", layer="routes")
route_buf_50.to_file(out_buf50, driver="GPKG", layer="route_buffer_50m")
route_buf_100.to_file(out_buf100, driver="GPKG", layer="route_buffer_100m")
school_bufs.to_file(out_school_bufs, driver="GPKG", layer="school_buffers")

for path, inputs, notes in (
    (out_routes, (assignments_path, schools_path, pwc_path), "Stage 5: shortest-path LineStrings (BNG) per LSOA-school pair."),
    (out_buf50, (out_routes,), f"Stage 5: routes buffered at {config.ROUTE_BUFFER_PRIMARY_M} m (primary)."),
    (out_buf100, (out_routes,), f"Stage 5: routes buffered at {config.ROUTE_BUFFER_SENSITIVITY_M} m (sensitivity #6)."),
    (out_school_bufs, (schools_path,), f"Stage 5: Euclidean school buffers at {config.BUFFER_DISTANCES_M} m."),
):
    audit.write_provenance_sidecar(
        audit.Provenance(output_path=path, inputs=inputs, notes=notes, random_seed=config.RANDOM_SEED)
    )
    print("wrote:", path.name)

## 6. Cross-check via osmnx (sensitivity #8)

Re-route a 10% random sample on a different routing engine (osmnx + networkx Dijkstra over the same OSM extract) and compare lengths. We expect a Pearson correlation > 0.99 if both engines see the same network.

In [ ]:
import networkx as nx

rng = np.random.default_rng(config.RANDOM_SEED)
sample_n = max(1, int(0.10 * len(routes)))
sample = routes.sample(n=sample_n, random_state=config.RANDOM_SEED).reset_index(drop=True)
print(f"Cross-check sample: {len(sample)} routes")

# Build a NetworkX graph from the same parsed walking-network parquet
G = nx.Graph()
G.add_nodes_from(data.nodes.index.tolist())
G.add_weighted_edges_from(
    zip(
        data.edges["from_id"].astype("int64").tolist(),
        data.edges["to_id"].astype("int64").tolist(),
        data.edges["length_m"].astype("float64").tolist(),
    ),
    weight="length_m",
)
print(f"NetworkX graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

In [ ]:
# Compute NetworkX Dijkstra for each sample pair; record length
nx_lengths = []
for _, r in sample.iterrows():
    o = int(r["origin_node"])
    d = int(r["dest_node"])
    try:
        L = nx.dijkstra_path_length(G, o, d, weight="length_m")
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        L = float("nan")
    nx_lengths.append(L)
sample["nx_length_m"] = nx_lengths
valid = sample["nx_length_m"].notna() & sample["length_m"].notna()
print(f"Routable in both engines: {valid.sum()} / {len(sample)}")
if valid.sum():
    pearson = sample.loc[valid, ["length_m", "nx_length_m"]].corr().iloc[0, 1]
    diff = sample.loc[valid, "length_m"] - sample.loc[valid, "nx_length_m"]
    print(f"Pearson r (pandana vs networkx lengths): {pearson:.4f}")
    print(f"Mean abs difference: {diff.abs().mean():.1f} m")
    print(f"Max abs difference : {diff.abs().max():.1f} m")

In [ ]:
# Save cross-check report
xcheck_path = config.AUDIT_LOGS / f"routing_cross_check_{TODAY}.csv"
sample[["lsoa21cd", "urn", "phase_cat", "length_m", "nx_length_m"]].to_csv(xcheck_path, index=False)
print("Wrote:", xcheck_path.relative_to(config.REPO_ROOT))

## 7. Quick-look — one Newcastle neighbourhood

Plot routes from one Newcastle LSOA to its assigned schools, with the 50 m buffer and the school's 800 m Euclidean buffer for context.

In [ ]:
import matplotlib.pyplot as plt

newc = pwc.merge(
    gpd.read_file(sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))[-1])[["lsoa21cd", "lad_code", "pop_school_age"]],
    on="lsoa21cd", how="left",
)
sample_lsoa = (
    newc.loc[newc["lad_code"] == "E08000021"].sort_values("pop_school_age", ascending=False).iloc[0]["lsoa21cd"]
)
rs = routes[routes["lsoa21cd"] == sample_lsoa]
rb = route_buf_50[route_buf_50["lsoa21cd"] == sample_lsoa]
ru = schools.loc[schools["urn"].isin(rs["urn"])]
sb = school_bufs[school_bufs["urn"].isin(rs["urn"]) & (school_bufs["distance_m"] == 800)]

fig, ax = plt.subplots(figsize=(8, 7))
sb.plot(ax=ax, facecolor="none", edgecolor="#3366cc", linewidth=0.8, label="800 m school buffer")
rb.plot(ax=ax, color="#cc0000", alpha=0.4, label="50 m route buffer")
rs.plot(ax=ax, color="#990000", linewidth=1.5, label="route")
ru.plot(ax=ax, color="black", markersize=15, label="school")
pwc.loc[pwc["lsoa21cd"] == sample_lsoa].plot(ax=ax, color="green", markersize=20, label="LSOA PWC")
ax.set_title(f"Routes from {sample_lsoa} (Newcastle, top-pop LSOA)")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## Done

Outputs ready for Stage 6 (`09_exposure_measurement.ipynb`):
- `data/processed/routes_<date>.gpkg`
- `data/processed/route_buffer_50m_<date>.gpkg` (primary)
- `data/processed/route_buffer_100m_<date>.gpkg` (sensitivity)
- `data/processed/school_euclidean_buffers_<date>.gpkg` (400/800/1600 m, long-format)
- `audit_logs/routing_cross_check_<date>.csv` (pandana vs networkx Dijkstra agreement)